In [ ]:
from sqlalchemy import create_engine

db_params = {
    'host': 'localhost',
    'database': 'postgres',
    'user': 'postgres',
    'password': 'postgres',
    'port': 5432
}

conn = create_engine(f"postgresql://{db_params['user']}:{db_params['password']}@{db_params['host']}:{db_params['port']}/{db_params['database']}")

## Number of Mutants per Project

In [ ]:
import pandas as pd

pd.read_sql_query("""
SELECT project_id, project_name, variant, sum(total) AS total
FROM mutation_results_by_project_variant_mutator
WHERE variant IN ('ORIGINAL', 'INITIAL')
GROUP BY project_id, project_name, variant
ORDER BY project_id, variant_order(variant)
""", conn)

## Number of Mutants per Project + Mutator

In [ ]:
import pandas as pd

pd.read_sql_query("""
SELECT project_id, project_name, variant, mutator, total
FROM mutation_results_by_project_variant_mutator
WHERE variant = 'INITIAL'
""", conn)

## Un-/Covered Mutants per Project + Variant

In [ ]:
import pandas as pd

df = pd.read_sql_query("""
SELECT project_id, project_name, variant, total, covered, uncovered, covered_pct, uncovered_pct
FROM mutation_results_by_project_variant
WHERE variant != 'ORIGINAL'
""", conn)

df = df.rename(columns={
  'covered_pct': 'Covered (%)',
  'uncovered_pct': 'Uncovered (%)',
})

df

## Percentage of Survived / Detected / ... Mutants per Project + Variant

In [ ]:
import pandas as pd

df = pd.read_sql_query("""
SELECT
    project_id, project_name, variant, 
    survived_of_covered_pct, killed_of_covered_pct, timed_out_of_covered_pct, memory_error_of_covered_pct,
    survived_of_covered_pct_diff, killed_of_covered_pct_diff, timed_out_of_covered_pct_diff, memory_error_of_covered_pct_diff
FROM mutation_results_by_project_variant
WHERE variant NOT IN ('ORIGINAL', 'BASELINE')
""", conn)

df = df.rename(columns={
    'survived_of_covered_pct': 'Survived (%)',
    'killed_of_covered_pct': 'Killed (%)',
    'timed_out_of_covered_pct': 'Timed-Out (%)',
    'memory_error_of_covered_pct': 'Memory Error (%)',
    'survived_of_covered_pct_diff': 'Survived (%) Diff.',
    'killed_of_covered_pct_diff': 'Killed (%) Diff.',
    'timed_out_of_covered_pct_diff': 'Timed-Out (%) Diff.',
    'memory_error_of_covered_pct_diff': 'Memory Error (%) Diff.',
})

df

## Percentage of Survived / Detected / ... Mutants per Project + Variant + Mutator

In [ ]:
import pandas as pd

df = pd.read_sql_query("""
SELECT
    project_id, project_name, variant, mutator,
    survived_of_covered_pct, killed_of_covered_pct, timed_out_of_covered_pct, memory_error_of_covered_pct,
    survived_of_covered_pct_diff, killed_of_covered_pct_diff, timed_out_of_covered_pct_diff, memory_error_of_covered_pct_diff
FROM mutation_results_by_project_variant_mutator
WHERE variant IN ('IMPROVED_1000_TRIES')
""", conn)

df = df.rename(columns={
    'survived_of_covered_pct': 'Survived (%)',
    'killed_of_covered_pct': 'Killed (%)',
    'timed_out_of_covered_pct': 'Timed-Out (%)',
    'memory_error_of_covered_pct': 'Memory Error (%)',
    'survived_of_covered_pct_diff': 'Survived (%) Diff.',
    'killed_of_covered_pct_diff': 'Killed (%) Diff.',
    'timed_out_of_covered_pct_diff': 'Timed-Out (%) Diff.',
    'memory_error_of_covered_pct_diff': 'Memory Error (%) Diff.',
})

df

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.colors as mcolors
from matplotlib.gridspec import GridSpec

def fetch_mutation_data(conn, variants=None):
    """Fetch and prepare mutation data from database."""
    query = "SELECT * FROM mutation_results_by_project_variant_mutator"
    if variants:
        variant_list = "', '".join(variants)
        query += f" WHERE variant IN ('{variant_list}')"

    results = pd.read_sql_query(query, conn)

    # Rename columns to match visualization expectations
    return results.rename(columns={
        'detected_pct': 'detection_rate',
        'detected_pct_diff': 'improvement'
    })

def get_sorted_variants(mutator_results, conn):
    """Get variants sorted by their order from database."""
    all_variants = sorted(
        mutator_results['variant'].unique(), 
        key=lambda v: pd.read_sql_query(f"SELECT variant_order('{v}')", conn).iloc[0, 0]
    )
    improvement_variants = [v for v in all_variants if v not in ['ORIGINAL', 'INITIAL', 'BASELINE']]
    return all_variants, improvement_variants

def create_variant_color_mapping(variants):
    """Create a consistent color mapping for variants using seaborn's colorblind palette."""
    # Use seaborn's colorblind-friendly palette
    color_palette = sns.color_palette("colorblind", n_colors=len(variants))

    # Convert RGB tuples to hex codes using matplotlib's color conversion
    hex_palette = [mcolors.to_hex(color) for color in color_palette]

    return {variant: hex_palette[i] for i, variant in enumerate(variants)}

def calculate_y_axis_limits(mutator_results, project_ids):
    """Calculate consistent y-axis limits across all plots."""
    detection_y_max = 0
    improvement_y_max = 0
    improvement_y_min = 0

    for project_id in project_ids:
        project_data = mutator_results[mutator_results['project_id'] == project_id]
        detection_y_max = max(detection_y_max, project_data['detection_rate'].max() * 1.1)

        project_improvements = project_data[project_data['improvement'].notna()]['improvement']
        if not project_improvements.empty:
            improvement_y_max = max(improvement_y_max, project_improvements.max() * 1.1)
            improvement_y_min = min(improvement_y_min, project_improvements.min() * 1.1)

    return detection_y_max, improvement_y_max, improvement_y_min

def plot_detection_rates(ax, project_data, all_variants, all_mutators, variant_colors):
    """Plot detection rate bars for a project."""
    mutator_positions = {mutator: i for i, mutator in enumerate(all_mutators)}
    width = 0.8 / len(all_variants)
    total_width = width * len(all_variants)
    group_offsets = -total_width/2 + width/2

    for variant_idx, variant in enumerate(all_variants):
        variant_data = project_data[project_data['variant'] == variant]
        detection_rates = {row['mutator']: row['detection_rate'] for _, row in variant_data.iterrows()}

        for mutator, pos in mutator_positions.items():
            rate = detection_rates.get(mutator, 0)
            if rate > 0:
                offset = group_offsets + width * variant_idx
                ax.bar(pos + offset, rate, width, color=variant_colors[variant])

def plot_improvements(ax, project_data, improvement_variants, all_mutators, variant_colors):
    """Plot improvement bars for a project."""
    if not improvement_variants:
        return

    mutator_positions = {mutator: i for i, mutator in enumerate(all_mutators)}
    width = 0.8 / len(improvement_variants)
    total_width = width * len(improvement_variants)
    group_offsets = -total_width/2 + width/2

    for variant_idx, variant in enumerate(improvement_variants):
        variant_data = project_data[project_data['variant'] == variant]
        improvements = {row['mutator']: row['improvement'] 
                       for _, row in variant_data.iterrows() 
                       if row['improvement'] is not None}

        for mutator, pos in mutator_positions.items():
            impr = improvements.get(mutator, 0)
            if impr != 0:
                offset = group_offsets + width * variant_idx
                ax.bar(pos + offset, impr, width, color=variant_colors[variant])

def configure_axis(ax, title, ylabel, x_min, x_max, y_min, y_max, all_mutators, show_xticklabels=False):
    """Configure axis properties."""
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_xticks(np.arange(len(all_mutators)))

    if show_xticklabels:
        ax.set_xticklabels(all_mutators, rotation=90, ha='center')
    else:
        ax.set_xticklabels([])

    ax.grid(axis='y', linestyle='--', alpha=0.7)

# Main visualization function
def visualize_mutation_results(conn, variants=None):
    """Create comprehensive visualization of mutation testing results."""
    # Set seaborn style for better aesthetics
    sns.set_style("whitegrid")

    # Fetch and prepare data
    mutator_results = fetch_mutation_data(conn, variants)
    all_variants, improvement_variants = get_sorted_variants(mutator_results, conn)
    all_mutators = sorted(mutator_results['mutator'].unique())
    project_ids = mutator_results['project_id'].unique()
    project_count = len(project_ids)

    # Calculate axis limits
    detection_y_max, improvement_y_max, improvement_y_min = calculate_y_axis_limits(
        mutator_results, project_ids)

    # Create color mapping using seaborn's colorblind palette
    variant_colors = create_variant_color_mapping(all_variants)

    # Create figure and grid
    fig = plt.figure(figsize=(18, 5 + 4 * project_count))
    gs = GridSpec(project_count, 2, width_ratios=[3, 2])

    # Set fixed x-axis limits
    x_min, x_max = -0.5, len(all_mutators) - 0.5

    # Create legend
    legend_handles = [plt.Rectangle((0, 0), 1, 1, color=variant_colors[variant]) 
                     for variant in all_variants]
    fig.legend(
        legend_handles, all_variants, loc='upper center',
        ncol=min(len(all_variants), 5), bbox_to_anchor=(0.5, 0.98), fontsize='small'
    )

    # Create plots for each project
    for i, project_id in enumerate(project_ids):
        project_data = mutator_results[mutator_results['project_id'] == project_id]
        if project_data.empty:
            continue

        # Get project name
        project_name = pd.read_sql_query(
            f"SELECT project_name({project_id})", conn).iloc[0, 0]

        # Create subplots
        ax1 = fig.add_subplot(gs[i, 0])  # Detection rate plot
        ax2 = fig.add_subplot(gs[i, 1])  # Improvement plot

        # Plot detection rates
        plot_detection_rates(ax1, project_data, all_variants, all_mutators, variant_colors)
        configure_axis(
            ax1, f'Detection Rate - Project ID: {project_id} - {project_name}',
            'Detection Rate (%)', x_min, x_max, 0, detection_y_max,
            all_mutators, show_xticklabels=(i == project_count - 1)
        )

        # Plot improvements
        plot_improvements(ax2, project_data, improvement_variants, all_mutators, variant_colors)
        configure_axis(
            ax2, f'Improvement - Project ID: {project_id} - {project_name}',
            'Improvement (%)', x_min, x_max, improvement_y_min, improvement_y_max,
            all_mutators, show_xticklabels=(i == project_count - 1)
        )
        ax2.axhline(y=0, color='k', linestyle='-', alpha=0.3)

    # Adjust layout
    plt.tight_layout(rect=[0, 0.03, 1, 0.92])
    plt.subplots_adjust(hspace=0.3, top=0.88)

    return fig

variants_to_plot = ['INITIAL', 'NAIVE_1000_TRIES', 'IMPROVED_1000_TRIES']
fig = visualize_mutation_results(conn, variants_to_plot)
plt.show()
